# Boundary Conditions and Convergence

This notebook bridges the NumPy baseline and the generated Cartesian wave project. It separates interior right-hand-side evaluation, ghost-zone fills, outer boundary handling, and convergence-factor reruns.

[Index](../index.ipynb) | Previous: [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb) | Next: [Finite-Difference Playground](finite_difference_playground.ipynb)

## Why This Matters

Boundary data and convergence tests decide whether a finite-difference evolution is numerically meaningful. This notebook separates interior updates, ghost zones, and reruns at different resolution.

## What You Will See

- Boundary-module capability checks.
- A generated Cartesian wave project inventory.
- Build, run, and convergence diagnostic output when tools are available.

## Table of Contents

1. [Interior updates and ghost zones](#Interior-updates-and-ghost-zones)
2. [Boundary infrastructure capabilities](#Boundary-infrastructure-capabilities)
3. [Temporary generated-project workspace](#Temporary-generated-project-workspace)
4. [Generated project inspection](#Generated-project-inspection)
5. [Build and convergence rerun](#Build-and-convergence-rerun)
6. [Convergence output summary](#Convergence-output-summary)
7. [Next notebooks](#Next-notebooks)

## Interior Updates and Ghost Zones

Finite-difference right-hand sides are evaluated on interior points. Before a stencil touches the edge of the domain, ghost zones must contain data consistent with the chosen boundary treatment. In generated projects, this leads to a recurring sequence:

1. Fill boundary data required by the next stencil evaluation.
2. Evaluate the right-hand side on valid interior points.
3. Advance the state with the Method of Lines.
4. Repeat at a second resolution to measure a convergence factor.

The Cartesian wave example uses its own generated Cartesian project. Curvilinear boundary-condition modules are separate infrastructure; the capability checks below only confirm that those modules are present.

## Boundary Infrastructure Capabilities

NRPy includes curvilinear boundary-condition modules for BHaH infrastructure. The Cartesian wave example should not be described as using those tools unless its generated project does so explicitly.

In [1]:
import importlib
import importlib.util

import nrpy


print("nrpy import succeeded")

boundary_modules = {
    "bcstruct_set_up": "register_CFunction_bcstruct_set_up",
    "apply_bcs_outerradiation_and_inner": "register_CFunction_apply_bcs_outerradiation_and_inner",
    "register_all": "register_C_functions",
}

base = "nrpy.infrastructures.BHaH.CurviBoundaryConditions"
for module_name, capability in boundary_modules.items():
    module = importlib.import_module(f"{base}.{module_name}")
    print(f"{module_name}: {capability} present =", hasattr(module, capability))

generator_name = "nrpy.examples.wave_equation_cartesian"
generator_spec = importlib.util.find_spec(generator_name)
print("Cartesian wave generator discovered:", generator_spec is not None)

nrpy import succeeded


bcstruct_set_up: register_CFunction_bcstruct_set_up present = True
apply_bcs_outerradiation_and_inner: register_CFunction_apply_bcs_outerradiation_and_inner present = True
register_all: register_C_functions present = True
Cartesian wave generator discovered: True


## Temporary Generated-Project Workspace

The generator is run in a fresh temporary directory so existing learner projects are not overwritten. The temporary workspace object is kept alive for later cells.

In [2]:
import subprocess
import sys
import tempfile
from pathlib import Path


workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_wave_boundary_")
workspace_path = Path(workspace_manager.name)
project_path = workspace_path / "project" / "wave_equation_cartesian"

print("Temporary workspace created.")
print("Project path will exist after generation:", project_path.exists())

Temporary workspace created.
Project path will exist after generation: False


## Generated Project Inspection

This cell runs the generator if it is discoverable, then reports a short file inventory. Generation failures are reported without stopping the notebook so learners can still read the boundary discussion.

In [3]:
generation_completed = False
if generator_spec is None:
    raise RuntimeError(f"Required generator not found: {generator_name}")
else:
    command = [sys.executable, "-m", generator_name]
    completed = subprocess.run(
        command,
        cwd=workspace_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Generator return code:", completed.returncode)
    if completed.stdout.strip():
        print("\n".join(completed.stdout.splitlines()[:12]))
    if completed.returncode == 0:
        generation_completed = True
    else:
        print("Generator stderr excerpt:")
        print("\n".join(completed.stderr.splitlines()[:12]))
        raise RuntimeError("Cartesian wave generator failed.")

if project_path.exists():
    inventory = sorted(path.name for path in project_path.iterdir())[:16]
    print("Generated project entries:")
    for name in inventory:
        print(" ", name)
else:
    print("Generated project directory is not present.")

Generator return code: 0
Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par
Generated project entries:
  BHaH_defines.h
  BHaH_function_prototypes.h
  Makefile
  MoL
  apply_bcs.c
  cmdline_input_and_parfile_parser.c
  commondata_struct_set_to_default.c
  diagnostics.c
  exact_solution_single_Cartesian_point.c
  griddata_free.c
  initial_data.c
  intrinsics
  main.c
  numerical_grids_and_timestep.c
  params_struct_set_to_default.c
  progress_indicator.c


## Build and Convergence Rerun

Building requires `make` and a C compiler. If either tool is unavailable, the notebook prints the commands to run later and continues.

In [4]:
import shutil


make_tool = shutil.which("make")
compiler_tool = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
can_build = project_path.exists() and make_tool is not None and compiler_tool is not None

print("make available:", make_tool is not None)
print("C compiler available:", compiler_tool is not None)

build_completed = False
run_completed = False
build_skipped_for_tools = False
if not project_path.exists():
    raise RuntimeError("Build cannot proceed because no generated project is available.")
elif not can_build:
    build_skipped_for_tools = True
    print("Build skipped. From the generated project directory, run: make -j2")
    print("Then run: ./wave_equation_cartesian")
    print("For a convergence rerun, run: ./wave_equation_cartesian 2.0")
else:
    build = subprocess.run(
        ["make", "-j2"],
        cwd=project_path,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print("Build return code:", build.returncode)
    if build.stdout.strip():
        print("\n".join(build.stdout.splitlines()[:12]))
    if build.returncode == 0:
        build_completed = True
    else:
        print("Build stderr excerpt:")
        print("\n".join(build.stderr.splitlines()[:12]))
        raise RuntimeError("Generated Cartesian wave project build failed.")

    executable = project_path / "wave_equation_cartesian"
    if build_completed and not executable.exists():
        raise RuntimeError("Generated Cartesian wave executable was not created.")
    for args in ([], ["2.0"]):
        run = subprocess.run(
            [f"./{executable.name}", *args],
            cwd=project_path,
            text=True,
            capture_output=True,
            timeout=180,
        )
        label = "default" if not args else "convergence factor 2.0"
        print(f"Run ({label}) return code:", run.returncode)
        if run.stdout.strip():
            print("\n".join(run.stdout.splitlines()[:8]))
        if run.returncode != 0 and run.stderr.strip():
            print("\n".join(run.stderr.splitlines()[:8]))
        if run.returncode != 0:
            raise RuntimeError(f"Generated Cartesian wave run failed: {label}")
        run_completed = True

make available: True
C compiler available: True


Build return code: 0
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c apply_bcs.c -o apply_bcs.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c cmdline_input_and_parfile_parser.c -o cmdline_input_and_parfile_parser.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c commondata_struct_set_to_default.c -o commondata_struct_set_to_default.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c diagnostics.c -o diagnostics.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c exact_solution_single_Cartesian_point.c -o exact_solution_single_Cartesian_point.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c griddata_free.c -o griddata_free.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c initial_data.c -o initial_data.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c main.c -o main.o
cc -std=gnu99 -O2 -march=native -g -Wall -I. -fopenmp  -c MoL/MoL_free_intermediate_stage_gfs.c -o MoL/MoL_free_intermediate_stage_gfs.o
cc -st

Run (default) return code: 0


Run (convergence factor 2.0) return code: 0


## Convergence Output Summary

The Cartesian project writes convergence diagnostics after successful runs. The exact filenames are discovered from the project directory rather than hard-coded into the lesson.

In [5]:
if project_path.exists():
    diagnostics = sorted(project_path.glob("out0d-conv_factor*.txt"))
    if diagnostics:
        for diagnostic in diagnostics:
            print("Diagnostic file:", diagnostic.name)
            print("\n".join(diagnostic.read_text(encoding="utf-8", errors="replace").splitlines()[:8]))
    elif run_completed:
        raise RuntimeError("No convergence diagnostic files were found after successful runs.")
    elif build_skipped_for_tools:
        print("No convergence diagnostics are available because build/run work was skipped.")
    else:
        print("No convergence diagnostic files were found.")
else:
    print("No generated project is available for diagnostic inspection.")

Diagnostic file: out0d-conv_factor1.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.991879e+00 3.991879e+00
1.562500e-01 4.061498e-08 2.140944e-05 3.983805e+00 3.983805e+00
4.687500e-01 4.029277e-07 2.042338e-05 3.919867e+00 3.919865e+00
6.250000e-01 7.161221e-07 1.953338e-05 3.864863e+00 3.864860e+00
7.812500e-01 1.107694e-06 1.838455e-05 3.795414e+00 3.795409e+00
9.375000e-01 1.567359e-06 1.698317e-05 3.712441e+00 3.712435e+00
1.250000e+00 2.637266e-06 1.345550e-05 3.510430e+00 3.510420e+00
1.406250e+00 3.214210e-06 1.134935e-05 3.393995e+00 3.393984e+00
Diagnostic file: out0d-conv_factor2.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.997967e+00 3.997967e+00
2.343750e-01 6.449302e-09 1.339274e-06 3.979733e+00 3.979733e+00
3.906250e-01 1.815270e-08 1.306709e-06 3.947547e+00 3.947547e+00
6.250000e-01 4.610504e-08 1.226542e-06 3.870305e+00 3.870304e+00
7.812500e-01 7.098276e-08 1.152656e-06 3.800507e+00 3.800506e+00
1.015625e+00 1.160006e-07 1.012224e-06 3.670669e+00 3.670668e+00
1.17

## Next Notebooks

[Finite-difference playground](finite_difference_playground.ipynb) isolates a single finite-difference kernel. [Start-to-finish Cartesian wave project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb) gives the complete generated-project workflow.

## Where This Leads

- [Method of Lines and Runge-Kutta](method_of_lines_and_rk.ipynb) reviews the prerequisite step.
- [Finite-Difference Playground](finite_difference_playground.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.